In [1]:
#claude task2.1
import pandas as pd
import re
from sklearn.metrics import f1_score
import time
import anthropic
client = anthropic.Anthropic(
    # defaults to os.environ.get("ANTHROPIC_API_KEY")
    api_key="api_key",
)

In [ ]:
#prompt 1 inference
'''
You are a NLP researcher. This is an argument unit mining task. You explain the reason and determine the given one sentence is claim or premise. If the sentence is a claim, you should return the class type 1. If the sentence is a premise, you should return the class type 0. Only return the number only
sentence:I mean, sometimes it's not that you came up with some brilliant strategy, it's just like really good work consistently over a long period of time.
Reason: The given sentence expresses an opinion or viewpoint about achieving success through consistent hard work over time, rather than relying on a single brilliant strategy. It presents a clear argument or stance on the topic, making it an argument unit.
output:1

sentence:So for example, we never participated as much in I'd call it non-developed markets, medium and small businesses with all of the sophisticated workloads.
Reason: The given sentence does not express a clear opinion or viewpoint, nor does it present an argument or stance on a particular issue. Instead, it seems to be stating a fact about the speaker's past participation in certain markets. Therefore, it does not meet the criteria for an argument unit
output: 0
'''
test = pd.read_json('/home/zengwesley/NTUNLP/Task 2.1/dev.json')
test_list=[]
for i in range(0, len(test)):
    message = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=100,
    temperature=0.0,
    system='''You are a NLP researcher. This is an argument unit mining task. You explain the reason and determine the given one sentence is claim or premise. If the sentence is a claim, you should return the class type 1. If the sentence is a premise, you should return the class type 0. Only return the number only
sentence:I mean, sometimes it's not that you came up with some brilliant strategy, it's just like really good work consistently over a long period of time.
Reason: The given sentence expresses an opinion or viewpoint about achieving success through consistent hard work over time, rather than relying on a single brilliant strategy. It presents a clear argument or stance on the topic, making it an argument unit.
output:1

sentence:So for example, we never participated as much in I'd call it non-developed markets, medium and small businesses with all of the sophisticated workloads.
Reason: The given sentence does not express a clear opinion or viewpoint, nor does it present an argument or stance on a particular issue. Instead, it seems to be stating a fact about the speaker's past participation in certain markets. Therefore, it does not meet the criteria for an argument unit
output: 0''',
    messages=[
        {"role": "user", "content": 'sentence:'+str(test.loc[i][0])}
    ]
)
    test_list.append(message.content)
    time.sleep(5)
    

#process the output into Kaggle format

#save the output
with open('prompt1_claude.txt', 'w') as file:
    file.writelines(str(item) + '\n' for item in test_list)

In [16]:
file_path = '/home/zengwesley/NTUNLP/Task 2.1/prompt1_claude.txt'
prompt1_claude=[]
with open(file_path, 'r') as file:
    for line in file:
        prompt1_claude.append(line)

In [17]:
  prompt1_claude

["[ContentBlock(text='Reason: The given sentence describes a process of modernization, where older workloads are being lifted and shifted, and the entire business process flow is being modernized. It does not present a clear argument or stance on a particular issue, but rather provides a factual description of an observed process. Therefore, it does not meet the criteria for an argument unit.\\n\\nOutput: 0', type='text')]\n",
 "[ContentBlock(text='Reason: The given sentence expresses a clear viewpoint or stance about maintaining a competitive number in the industry. It presents an argument or claim about the importance of keeping this number competitive, making it an argument unit.\\n\\nOutput: 1', type='text')]\n",
 "[ContentBlock(text='Reason: The given sentence is a simple statement about the performance of a quarter, without presenting any clear argument or stance. It does not express an opinion or make a claim, but rather states a factual observation. Therefore, it does not meet 

In [18]:
#extract the first number from each line

def match_first_number(string):
    numbers = re.findall(r'\d+', string)
    if numbers:
        if numbers[0] == 0 or 1:
            return numbers[0]
    else:
        return 0

y_pred=[]
for i in prompt1_claude:
    first_number = match_first_number(i)
    if first_number ==str(0) or first_number ==str(1):
        y_pred.append(int(first_number))
    else:
        y_pred.append(int(0))


In [23]:
#second prompt inference
label=pd.read_json('/home/zengwesley/NTUNLP/Task 2.1/dev.json')
gpt_result=y_pred
gt_result=label[1]

In [24]:
from sklearn.metrics import f1_score
micro_f1 = f1_score(gt_result, gpt_result, average='micro')
print(f'Micro F1 Score: {micro_f1}')

macro_f1 = f1_score(gt_result, gpt_result, average='macro')
print(f'Macro F1 Score: {macro_f1}')

weighted_f1 = f1_score(gt_result, gpt_result, average='weighted')
print(f'Weighted F1 Score: {weighted_f1}')

Micro F1 Score: 0.6016511867905057
Macro F1 Score: 0.6004559088694998
Weighted F1 Score: 0.6015158723088825


In [ ]:
#prompt 2 inference
'''You will be performing argument unit mining on a given sentence to determine if it is a claim or a premise. 
Carefully read the sentence and think about whether it expresses an opinion, viewpoint, or stance on an issue (making it a claim) or if it simply states a fact or supporting evidence (making it a premise).
Write out your reasoning for the classification inside reasoning tags.
Finally, if the sentence is a claim, output number 1 . If the sentence is a premise, output number 0
Do not output the reason. Only output the number only.  '''

test = pd.read_json('/home/zengwesley/NTUNLP/Task 2.1/dev.json')
test_list=[]
for i in range(0, len(test)):
    message = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=100,
    temperature=0.0,
    system='''You will be performing argument unit mining on a given sentence to determine if it is a claim or a premise. 
Carefully read the sentence and think about whether it expresses an opinion, viewpoint, or stance on an issue (making it a claim) or if it simply states a fact or supporting evidence (making it a premise).
Write out your reasoning for the classification inside reasoning tags.
Finally, if the sentence is a claim, output number 1 . If the sentence is a premise, output number 0
Do not output the reason. Only output the number only.  ''',
    messages=[
        {"role": "user", "content": 'sentence:'+str(test.loc[i][0])}
    ]
)
    test_list.append(message.content)
    time.sleep(5)
    

#process the output into Kaggle format

#save the output
test_list.to_csv('prompt2_claude.csv', index=False)

In [26]:
#read prompt 2
file_path = '/home/zengwesley/NTUNLP/Task 2.1/prompt2_claude.txt'
prompt2_claude=[]
with open(file_path, 'r') as file:
    for line in file:
        prompt2_claude.append(line)

In [27]:
prompt2_claude

["[ContentBlock(text='<reasoning>\\nThe given sentence describes the process of modernizing business operations, including lifting and shifting older workloads. It does not express an opinion or stance on an issue, but rather states a factual observation about the actions being taken. Therefore, this sentence is a premise, not a claim.\\n</reasoning>\\n0', type='text')]\n",
 "[ContentBlock(text='<reasoning>\\nThe given sentence expresses an opinion or viewpoint about a specific number being incredibly competitive in the industry, and the desire to maintain that competitiveness. This suggests that the sentence is making a claim rather than simply stating a fact or supporting evidence.\\n</reasoning>\\n1', type='text')]\n",
 "[ContentBlock(text='<reasoning>\\nThe given sentence does not express an opinion, viewpoint, or stance on an issue. It simply states a fact about a quarter being strong, without making any claims. Therefore, this sentence is a premise.\\n</reasoning>\\n0', type='tex

In [28]:
#extract first number from each line
y_pred=[]
for i in prompt2_claude:
    first_number = match_first_number(i)
    if first_number ==str(0) or first_number ==str(1):
        y_pred.append(int(first_number))
    else:
        y_pred.append(int(0))

In [30]:
label=pd.read_json('/home/zengwesley/NTUNLP/Task 2.1/dev.json')
gpt_result=y_pred
gt_result=label[1]

In [33]:
#prompt 2 F1 score
micro_f1 = f1_score(gt_result, gpt_result, average='micro')
print(f'Micro F1 Score: {micro_f1}')

macro_f1 = f1_score(gt_result, gpt_result, average='macro')
print(f'Macro F1 Score: {macro_f1}')

weighted_f1 = f1_score(gt_result, gpt_result, average='weighted')
print(f'Weighted F1 Score: {weighted_f1}')

Micro F1 Score: 0.6273637374860956
Macro F1 Score: 0.6083664825740542
Weighted F1 Score: 0.6124921490448512
